In [4]:
import numpy as np
import math
import re
import itertools
import pandas as pd
import time
import os
import pickle
import copy
from Functions import *
from Fast_functions import *
import cmcrameri.cm as cmc

In [5]:
diff=3
max_checks=1000
n_compounds=100
scrambler={1:np.arange(n_compounds)}
for j in range(2,diff+1):
    scrambler.update({j:np.array(list(itertools.combinations(np.arange(n_compounds),j)))})

In [6]:
def decode_precomp(well_assigner:np.array, differentiate:int, 
                   scrambler:dict, readout:np.ndarray, max_differentiate=-1, **kwargs) -> list:
    if differentiate==0:
        return(True,well_assigner, np.array([1]*well_assigner.shape[0]))
    N=well_assigner.shape[0]
    sc_list=np.arange(N).tolist()
    for i in range(differentiate):
        diff=i+1
        if diff ==1:
            full_well_assigner=well_assigner.copy()
        else:
            this_sc=scrambler[diff]
            #print(this_sc)
            #print(well_assigner)
            #print(diff)
            full_well_assigner=np.concatenate((full_well_assigner,np.any(well_assigner[this_sc], axis=1)))
            sc_list.extend(this_sc.tolist())
    #outcomes,_=np.unique(full_well_assigner, axis=0, return_counts=True)
    idxs = np.all(readout == full_well_assigner, axis=1)
    return list(itertools.compress(sc_list,idxs))
        

In [ ]:

def fast_decode(well_assigner:np.array, differentiate:int, readout:np.ndarray, 
                max_checks=1e4, **kwargs):
    WA=well_assigner
    n_pools=WA.shape[1]


    if np.max(readout)>1 or len(readout)!=n_pools:
        readout_bin_ls = [1 if i in readout else 0 for i in range(n_compounds)]
        readout_bl=np.array(readout_bin_ls)
    else:
        readout_bl=readout
    mask = ~np.any((WA == 1) & (readout_bl == 0), axis=1)
                    #msg+=f'boolean readout {readout_bl}<br>'
    original_indices = np.where(mask)[0]  # Get original row indices
    filtered_WA = WA[mask]
    n_compounds=filtered_WA.shape[0]
    if n_compounds<differentiate:
        differentiate=n_compounds
    if n_compounds<2:
        if n_compounds==1:
            decoded=[original_indices[0]] 
        else:
            decoded=[]

    else:
        MC=0
        MP=1
        ls_combs=[]
        ls_diffs=[]
        difo=differentiate
        while (MC<max_checks or MP>mp) and difo>0:
            MCI=math.comb(n_compounds,differentiate)
            MC+=MCI
            mp=MCI/MC
            ls_combs.append(MCI)
            ls_diffs.append(difo)
            difo-=1
        if MC>max_checks:
            decoded = [int(original_indices[idx]) for idx in range(len(original_indices))]
        
        else:
            scrambler={1:np.arange(n_compounds)}
            for j in range(2,differentiate+1):
                scrambler.update({j:np.array(list(itertools.combinations(np.arange(n_compounds),j)))})
            decoded_pre=decode_precomp(well_assigner=filtered_WA, differentiate= differentiate, scrambler=scrambler, 
                    readout=readout_bl)
            # Map filtered indices back to original indices
            decoders = [combination if isinstance(combination, list) else [combination] for combination in decoded_pre]
            decoded = [[int(original_indices[idx]) for idx in combination] for combination in decoders]
        
    return decoded




In [8]:
def is_consistent_precomp(well_assigner:np.array, differentiate:int, scrambler:dict) -> list:
    if differentiate==0:
        return(True,well_assigner, np.array([1]*well_assigner.shape[0]))
    N=well_assigner.shape[0]
    for i in range(differentiate):
        diff=i+1
        if diff ==1:
            full_well_assigner=well_assigner.copy()
        else:
            this_sc=scrambler[diff]
            #print(this_sc)
            #print(well_assigner)
            #print(diff)
            
            full_well_assigner=np.concatenate((full_well_assigner,np.any(well_assigner[this_sc], axis=1)))
    _, counts=np.unique(full_well_assigner, axis=0, return_counts=True)
    if len(counts)<full_well_assigner.shape[0]:
        return(False, full_well_assigner, counts)
    elif len(counts)==full_well_assigner.shape[0]:
        return(True,full_well_assigner, counts)
    else:
        print("Something is fishy")
        return(-1)
    
def mean_metrics_precomp(well_assigner, differentiate, scrambler, **kwargs):
    BT=well_assigner.shape[1]
    _,_, counts= is_consistent_precomp(well_assigner, differentiate, scrambler) 
    ET=extra_tests(counts)  
    ET =ET if ET<well_assigner.shape[0] else well_assigner.shape[0]
    ER=np.sum(counts[counts>1])/np.sum(counts)
    rounds=ER+1
    p_check=np.round(ER*100)
    return BT+ET, ET,  rounds, p_check

def mean_metrics_fast(well_assigner, differentiate, max_checks=1e4, scaler=1, mp=1e-5, **kwargs):
    BT=well_assigner.shape[1]
    n_compounds=well_assigner.shape[0]
    MC=0
    MP=1
    ls_combs=[]
    ls_diffs=[]
    difo=differentiate
    while (MC<max_checks or MP>mp) and difo>0:
        MCI=math.comb(n_compounds,differentiate)
        MC+=MCI
        mp=MCI/MC
        ls_combs.append(MCI)
        ls_diffs.append(difo)
        difo-=1
        

    if MC>max_checks:
        counts=[]
        probi=np.array(ls_combs, dtype=float)
        probi/=np.sum(probi)
        differis=np.random.choice(ls_diffs, int(max_checks*scaler), p=probi)
        #differo=differentiate
        #for _ in range(int(max_checks*scaler)):
        for differo in differis:
            rnd_pos=np.random.choice(np.arange(n_compounds), differo, replace=False)
            readout=np.any(well_assigner[rnd_pos], axis=0)
            decoded=fast_decode(well_assigner=well_assigner, differentiate=differo, 
                                readout=readout, max_checks=int(max_checks/10+5))
            counts.append(len(decoded))
        counts=np.array(counts)
        ET=np.sum(counts-1)/len(counts)
        ET =ET if ET<well_assigner.shape[0] else well_assigner.shape[0]
        ER=np.sum(counts>1)/np.sum(counts>0)
        rounds=ER+1
        p_check=np.round(ER*100)
        return BT+ET, ET,  rounds, p_check
    else:
        scrambler={1:np.arange(n_compounds)}
        for j in range(2,differentiate+1):
            scrambler.update({j:np.array(list(itertools.combinations(np.arange(n_compounds),j)))})
        
        return mean_metrics_precomp(well_assigner, differentiate, scrambler, **kwargs)
                


In [9]:
differo=15
compi=5000

In [10]:
WA_mat=assign_wells_mat(n_compounds=compi)

In [11]:
rnd_pos=np.random.choice(np.arange(compi), differo, replace=True)

In [12]:
readout=np.any(WA_mat[rnd_pos], axis=0)

In [13]:
fast_decode(well_assigner=WA_mat, differentiate=differo, 
                                readout=readout, max_checks=5)

(5000, 142)


[497,
 498,
 500,
 501,
 505,
 513,
 518,
 530,
 531,
 533,
 560,
 562,
 566,
 781,
 782,
 784,
 785,
 789,
 797,
 802,
 814,
 815,
 817,
 844,
 846,
 850,
 852,
 853,
 855,
 856,
 860,
 868,
 873,
 885,
 886,
 888,
 915,
 917,
 921,
 1349,
 1350,
 1352,
 1353,
 1357,
 1365,
 1370,
 1382,
 1383,
 1385,
 1412,
 1414,
 1418,
 1491,
 1492,
 1494,
 1495,
 1499,
 1507,
 1512,
 1524,
 1525,
 1527,
 1554,
 1556,
 1560,
 1562,
 1563,
 1565,
 1566,
 1570,
 1578,
 1583,
 1595,
 1596,
 1598,
 1625,
 1627,
 1631,
 1704,
 1705,
 1707,
 1708,
 1712,
 1720,
 1725,
 1737,
 1738,
 1740,
 1767,
 1769,
 1773,
 1988,
 1989,
 1991,
 1992,
 1996,
 2004,
 2009,
 2021,
 2022,
 2024,
 2051,
 2053,
 2057,
 2272,
 2273,
 2275,
 2276,
 2280,
 2288,
 2293,
 2305,
 2306,
 2308,
 2335,
 2337,
 2341,
 2769,
 2770,
 2772,
 2773,
 2777,
 2785,
 2790,
 2802,
 2803,
 2805,
 2832,
 2834,
 2838,
 3195,
 3196,
 3198,
 3199,
 3203,
 3211,
 3216,
 3228,
 3229,
 3231,
 3258,
 3260,
 3264,
 3337,
 3338,
 3340,
 3341,
 3345,
 33

In [11]:
mean_metrics_fast(WA_mat, differo, max_checks=10, scaler=0.3)

(332.0, 190.0, 2.0, 100.0)

In [ ]:
mean_metrics_fast(WA_mat, differo)

In [ ]:
scramblero={1:np.arange(compi)}
for j in range(2,differo+1):
    scramblero.update({j:np.array(list(itertools.combinations(np.arange(compi),j)))})
mean_metrics_precomp(WA_mat, differo, scramblero)


In [ ]:
2/9


In [ ]:
[math.comb(compi,i+1) for i in range(differo)]

In [ ]:
rout=np.sum(WA_mat[np.array([1])], axis=0).astype(bool)

In [ ]:
fast_decode(well_assigner=WA_mat, differentiate=1, 
                            readout=rout)

In [ ]:
n_compounds=85
diff=2

In [ ]:
obji=[1,2,3,4,5]
[list(x) for x in itertools.combinations(obji, 2)]

In [ ]:
scrambler={1:np.arange(n_compounds)}
for j in range(2,diff+1):
    scrambler.update({j:np.array(list(itertools.combinations(np.arange(n_compounds),j)))})

In [ ]:
WA_mat[scrambler[2]].shape

In [ ]:
math.comb(n_compounds,50)

In [ ]:
stirl_binom_extra_approx(n_compounds,50)